#plots for report

In [ ]:
"""
Consensus Analysis Script — Neha's EGFR Binder Data
Adapted from postyr's analysis script.

Inputs:
    - contacts parquet:      rank1_target_contacts_long_5A.parquet
    - pocket RMSD parquet:   pairwise_pocket_align_rmsd.parquet
    - binder RMSD parquet:   pairwise_binder_ca_rmsd.parquet
    - binding labels CSV:    proteinbase_cleaned_data_28_01_2026.csv  (long format)

Key differences from postyr's script:
    - Binding labels are stored in long-format CSV (metric == "binding")
    - Pocket RMSD column is `pocket_binder_ca_rmsd`
    - Binder RMSD column is `binder_ca_rmsd`
    - Full 3-feature consensus score: Jaccard_z − BinderRMSD_z − PocketRMSD_z
    - Models: AF2, AF3, Boltz-2, Chai-1, OF2, OF3, Protenix, HelixFold3
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import colors
from pathlib import Path
from itertools import combinations
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

# ─────────────────────────────────────────────
# GLOBAL PLOT STYLE
# ─────────────────────────────────────────────
sns.set_theme(
    style="ticks",
    context="paper",
    font_scale=1.2
)

plt.rcParams.update({
    "figure.dpi": 300,
    "savefig.dpi": 600,
    "axes.linewidth": 1.0,
    "axes.labelsize": 12,
    "axes.titlesize": 13,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "pdf.fonttype": 42,
    "ps.fonttype": 42
})

# ─────────────────────────────────────────────
# 0. PATHS — update these if needed
# ─────────────────────────────────────────────
CONTACTS_PATH    = r"C:\\Users\\nehaj\\OneDrive\\Desktop\\Block_4_new_project\\a_1+2\\rank_target_contacts_long_5A.parquet"
POCKET_RMSD_PATH = r"C:\\Users\\nehaj\\OneDrive\\Desktop\\Block_4_new_project\\a_1+2\\pairwise_pocket_align_rmsd_merged.parquet"
BINDER_RMSD_PATH = r"C:\\Users\\nehaj\\OneDrive\\Desktop\\Block_4_new_project\\a_1+2\\pairwise_binder_ca_rmsd_merged.parquet"
BINDING_CSV_PATH = r"C:\\Users\\nehaj\\OneDrive\\Desktop\\Block_4_new_project\\a_1+2\\contest_1_2_exp_data.csv"

# ─────────────────────────────────────────────
# 1. LOAD DATA
# ─────────────────────────────────────────────
print("Loading data...")

contacts_df    = pd.read_parquet(CONTACTS_PATH)
pocket_rmsd_df = pd.read_parquet(POCKET_RMSD_PATH)
binder_rmsd_df = pd.read_parquet(BINDER_RMSD_PATH)

# Long-format CSV → extract binding label per sequence
raw_csv = pd.read_csv(BINDING_CSV_PATH)
binding_labels = (
    raw_csv[raw_csv["metric"] == "binding"][["sequence", "value"]]
    .drop_duplicates(subset="sequence")
    .rename(columns={"value": "binding"})
)
# Coerce to bool — handles "True"/"False" strings or 0/1
binding_labels["binding"] = binding_labels["binding"].map(
    lambda x: str(x).strip().lower() in ("true", "1", "yes")
)

print(f"  Sequences with binding labels: {len(binding_labels)}")
print(f"  Binding True:  {binding_labels['binding'].sum()}")
print(f"  Binding False: {(~binding_labels['binding']).sum()}")

# ─────────────────────────────────────────────
# 2. STANDARDISE COLUMN NAMES
# ─────────────────────────────────────────────
# Pocket RMSD: rename RMSD column to a clean internal name
pocket_rmsd_df = pocket_rmsd_df.rename(columns={
    "pocket_binder_ca_rmsd": "rmsd_pocket_aligned"
})
# Filter to ok rows only
pocket_rmsd_df = pocket_rmsd_df[pocket_rmsd_df["status_pocket"] == "ok"].copy()

# Binder RMSD: rename RMSD column to a clean internal name
binder_rmsd_df = binder_rmsd_df.rename(columns={
    "binder_ca_rmsd": "rmsd_binder_aligned"
})
# Filter to ok rows only
binder_rmsd_df = binder_rmsd_df[binder_rmsd_df["status"] == "ok"].copy()

# ─────────────────────────────────────────────
# 3. MODEL ORDER
# ─────────────────────────────────────────────
method_order = ["AF2", "AF3", "Boltz-2", "Chai-1", "OF2", "OF3", "Protenix", "HelixFold3","SeedFold"]

# ─────────────────────────────────────────────
# 4. MERGE BINDING LABELS
# ─────────────────────────────────────────────
contacts_merged = contacts_df.merge(binding_labels, on="sequence", how="left") \
    if "sequence" in contacts_df.columns else \
    contacts_df.merge(binding_labels, left_on="binder_sequence", right_on="sequence", how="left")

pocket_merged = pocket_rmsd_df.merge(
    binding_labels, left_on="binder_sequence", right_on="sequence", how="left"
).drop(columns=["sequence"], errors="ignore")

binder_merged = binder_rmsd_df.merge(
    binding_labels, left_on="binder_sequence", right_on="sequence", how="left"
).drop(columns=["sequence"], errors="ignore")

# Keep only Miniprotein + Peptide design classes if available
if "design_class" in contacts_merged.columns:
    plot_target = contacts_merged[
        contacts_merged["design_class"].isin(["Miniprotein", "Peptide"])
    ].copy()
else:
    print("  Warning: no design_class column in contacts — using all rows")
    plot_target = contacts_merged.copy()

if "design_class" in pocket_merged.columns:
    plot_pocket = pocket_merged[
        pocket_merged["design_class"].isin(["Miniprotein", "Peptide"])
    ].copy()
else:
    plot_pocket = pocket_merged.copy()

plot_binder = binder_merged.copy()  # binder RMSD has no design_class column

# ─────────────────────────────────────────────
# 5. JACCARD INDEX (interface overlap)
# ─────────────────────────────────────────────
print("\nComputing Jaccard indices...")

contact_sets = (
    plot_target
    .groupby(["binder_sequence", "method", "binding"])["target_resseq"]
    .apply(lambda x: set(x.dropna()))
    .reset_index(name="contact_set")
)

rows = []
for binder, group in contact_sets.groupby("binder_sequence"):
    methods = group["method"].values
    sets    = group["contact_set"].values
    binding = group["binding"].iloc[0]

    for (m1, s1), (m2, s2) in combinations(zip(methods, sets), 2):
        union = s1 | s2
        jaccard = len(s1 & s2) / len(union) if union else 0.0
        rows.append({
            "binder_sequence": binder,
            "method_1": m1,
            "method_2": m2,
            "jaccard": jaccard,
            "binding": binding
        })

jaccard_df = pd.DataFrame(rows)
print(f"  Jaccard pairs computed: {len(jaccard_df)}")

# ── 5a. Median Jaccard heatmap (all binders) ──────────────────────────────────
def make_jaccard_matrix(df):
    matrix = df.groupby(["method_1", "method_2"])["jaccard"].median().unstack()
    matrix = matrix.combine_first(matrix.T)
    matrix = matrix.reindex(index=method_order, columns=method_order)
    for m in method_order:
        if m in matrix.index:
            matrix.loc[m, m] = 1.0
    return matrix

norm = colors.PowerNorm(gamma=0.5)

jac_matrix = make_jaccard_matrix(jaccard_df)

plt.figure(figsize=(8, 6))
ax= sns.heatmap(
    jac_matrix, annot=True, fmt=".2f", cmap="mako_r",
    vmin=0.4, vmax=1, square=True, norm=norm,
    cbar_kws={"label": "Median Jaccard index"},
    linewidths=0.5, linecolor="white", annot_kws={"size": 8}
)
ax.set_xlabel("Model 1")
ax.set_ylabel("Model 2")
plt.title("Pairwise Interface consensus \n (Jaccard index)")
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
sns.despine(left=True, bottom=True)
plt.tight_layout()
plt.savefig("Figure_1_jaccard_heatmap_all.pdf", dpi=600, bbox_inches="tight")
plt.show()

# ── 5b. Binding vs Non-binding Jaccard heatmaps ───────────────────────────────
binding_jac    = make_jaccard_matrix(jaccard_df[jaccard_df["binding"] == True])
nonbinding_jac = make_jaccard_matrix(jaccard_df[jaccard_df["binding"] == False])

fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharex=True, sharey=False)
for ax, matrix, title in zip(axes,
                              [binding_jac, nonbinding_jac],
                              ["Binding = True", "Binding = False"]):
    sns.heatmap(
        matrix, ax=ax, annot=True, fmt=".2f", cmap="mako_r",
        vmin=0.4, vmax=1, square=True, norm=norm,
        cbar=(ax == axes[1]),
        cbar_kws={"label": "Median Jaccard index"} if ax == axes[1] else {},
        linewidths=0.5, linecolor="white", annot_kws={"size": 8}
    )
    ax.set_title(title)
    ax.set_xlabel("Model 1"); ax.set_ylabel("Model 2")
    ax.tick_params(axis="x", rotation=45)
    ax.tick_params(axis="y", rotation=0)
    sns.despine(ax=ax, left=True, bottom=True)
fig.suptitle("Pairwise interface consensus(Jaccard Index) \n Split by Experimental Binding", y=0.95)
plt.tight_layout()
plt.savefig("Figure_2_jaccard_heatmap_split.pdf", dpi=600, bbox_inches="tight")
plt.show()

# ── 5c. Per-model median Jaccard vs binding ───────────────────────────────────
jaccard_long = pd.concat([
    jaccard_df.rename(columns={"method_1": "method", "method_2": "other"}),
    jaccard_df.rename(columns={"method_2": "method", "method_1": "other"})
]).reset_index(drop=True)

model_jaccard = (
    jaccard_long
    .groupby(["binder_sequence", "method"])["jaccard"]
    .median()
    .reset_index()
    .rename(columns={"jaccard": "median_jaccard"})
    .merge(binding_labels, left_on="binder_sequence", right_on="sequence", how="left")
    .drop(columns=["sequence"], errors="ignore")
)

palette = {False: "#4C72B0", True: "#DD8452"}
g = sns.catplot(
    data=model_jaccard,
    x="binding", y="median_jaccard",
    col="method", kind="boxen",
    col_wrap=4, height=3, sharey=True,
    hue="binding", col_order=[m for m in method_order if m in model_jaccard["method"].unique()],
    palette=palette, legend=False
)
# Apply linewidth and k_depth to each boxenplot axes
for ax in g.axes.flat:
    for artist in ax.get_children():
        if hasattr(artist, 'set_linewidth'):
            artist.set_linewidth(1)
g.set_titles("{col_name}")
g.set_axis_labels("Experimental binding", "Median Jaccard index")
plt.suptitle("Median Jaccard per model vs experimental binding", y=1.02)
sns.despine()
plt.tight_layout()
plt.savefig("Figure_3_jaccard_per_model.pdf", dpi=600, bbox_inches="tight")
plt.show()



In [ ]:

# ─────────────────────────────────────────────
# 6. POCKET RMSD HEATMAPS
# ─────────────────────────────────────────────
print("\nGenerating pocket RMSD heatmaps...")

def make_rmsd_matrix(df, col="rmsd_pocket_aligned"):
    matrix = df.groupby(["reference_method", "query_method"])[col].median().unstack()
    matrix = matrix.combine_first(matrix.T)
    matrix = matrix.reindex(index=method_order, columns=method_order)
    for m in method_order:
        if m in matrix.index:
            matrix.loc[m, m] = 0.0
    return matrix

# ── 6a. Overall pocket RMSD heatmap ──────────────────────────────────────────
rmsd_matrix = make_rmsd_matrix(plot_pocket)

plt.figure(figsize=(8, 6))
ax=sns.heatmap(
    rmsd_matrix, annot=True, fmt=".2f", cmap="mako",
    square=True, mask=rmsd_matrix.isna(),
    cbar_kws={"label": "Median pocket-aligned RMSD (Å)"},
    linewidths=0.5, linecolor="white", annot_kws={"size": 8}
)
ax.set_xlabel("Moving model")
ax.set_ylabel("Reference Model")
plt.title("Pair-wise Pocket-aligned RMSD \n (Docking Site Consensus)")
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
sns.despine(left=True, bottom=True)
plt.tight_layout()
plt.savefig("Figure_4_pocket_rmsd_heatmap_all.pdf", dpi=600, bbox_inches="tight")
plt.show()

# ── 6b. Binding vs Non-binding pocket RMSD ───────────────────────────────────
binding_rmsd    = make_rmsd_matrix(plot_pocket[plot_pocket["binding"] == True])
nonbinding_rmsd = make_rmsd_matrix(plot_pocket[plot_pocket["binding"] == False])

vmin = min(binding_rmsd.min().min(), nonbinding_rmsd.min().min())
vmax = max(binding_rmsd.max().max(), nonbinding_rmsd.max().max())

fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharex=True, sharey=False)
for ax, matrix, title in zip(axes,
                              [binding_rmsd, nonbinding_rmsd],
                              ["Binding = True", "Binding = False"]):
    sns.heatmap(
        matrix, ax=ax, annot=True, fmt=".2f", cmap="mako",
        square=True, mask=matrix.isna(),
        vmin=vmin, vmax=vmax,
        cbar=(ax == axes[1]),
        cbar_kws={"label": "Median pocket RMSD (Å)"} if ax == axes[1] else {},
        linewidths=0.5, linecolor="white", annot_kws={"size": 8}
    )
    ax.set_title(title)
    ax.set_xlabel("Moving Model"); ax.set_ylabel("Reference Model")
    ax.tick_params(axis="x", rotation=45)
    ax.tick_params(axis="y", rotation=0)
    sns.despine(ax=ax, left=True, bottom=True)
fig.suptitle("Pair-wise Pocket-aligned RMSD \n Split by Experimental Binding", y=0.95)
plt.tight_layout()
plt.savefig("Figure_5_pocket_rmsd_heatmap_split.pdf", dpi=600, bbox_inches="tight")
plt.show()

# ── 6c. Per-method RMSD distribution (boxen) ─────────────────────────────────
long_rmsd = pd.concat([
    plot_pocket[["reference_method", "rmsd_pocket_aligned"]].rename(columns={"reference_method": "method"}),
    plot_pocket[["query_method",     "rmsd_pocket_aligned"]].rename(columns={"query_method":     "method"})
]).reset_index(drop=True)

plt.figure(figsize=(10, 5))
sns.boxenplot(data=long_rmsd, x="method", y="rmsd_pocket_aligned",
              order=[m for m in method_order if m in long_rmsd["method"].unique()],
              color="#8172B2", linewidth=1, k_depth="proportion")
plt.xlabel("Model")
plt.ylabel("Pocket-aligned binder RMSD (Å)")
plt.title("Distribution of pairwise pocket aligned RMSD")
plt.xticks(rotation=45, ha="right")
sns.despine()
plt.tight_layout()
plt.savefig("Figure_6_pocket_rmsd_per_method.pdf", dpi=600, bbox_inches="tight")
plt.show()

# ─────────────────────────────────────────────
# 6d. BINDER-ALIGNED RMSD HEATMAPS
# (aligned on binder Cα — measures fold consensus,
#  i.e. do models agree on the binder's own structure?)
# ─────────────────────────────────────────────
print("\nGenerating binder-aligned RMSD heatmaps...")

def make_binder_rmsd_matrix(df):
    # binder parquet uses method_1 / method_2 (not reference/query)
    matrix = df.groupby(["method_1", "method_2"])["rmsd_binder_aligned"].median().unstack()
    matrix = matrix.combine_first(matrix.T)
    matrix = matrix.reindex(index=method_order, columns=method_order)
    for m in method_order:
        if m in matrix.index:
            matrix.loc[m, m] = 0.0
    return matrix

# Overall binder RMSD heatmap
binder_rmsd_matrix = make_binder_rmsd_matrix(plot_binder)

plt.figure(figsize=(8, 6))
sns.heatmap(
    binder_rmsd_matrix, annot=True, fmt=".2f", cmap="mako",
    square=True, mask=binder_rmsd_matrix.isna(),
    cbar_kws={"label": "Median binder-aligned RMSD (Å)"},
    linewidths=0.5, linecolor="white", annot_kws={"size": 8}
)
plt.title("Pair-wise Binder-aligned Cα RMSD \n(fold consensus)")
plt.xlabel("Model 1"); plt.ylabel("Model 2")
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
sns.despine(left=True, bottom=True)
plt.tight_layout()
plt.savefig("Figure_7_binder_rmsd_heatmap_all.pdf", dpi=600, bbox_inches="tight")
plt.show()

# Split by binding outcome
binding_brmsd    = make_binder_rmsd_matrix(plot_binder[plot_binder["binding"] == True])
nonbinding_brmsd = make_binder_rmsd_matrix(plot_binder[plot_binder["binding"] == False])

vmin_b = min(binding_brmsd.min().min(), nonbinding_brmsd.min().min())
vmax_b = max(binding_brmsd.max().max(), nonbinding_brmsd.max().max())

fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharex=True, sharey=False)
for ax, matrix, title in zip(axes,
                              [binding_brmsd, nonbinding_brmsd],
                              ["Binding = True", "Binding = False"]):
    sns.heatmap(
        matrix, ax=ax, annot=True, fmt=".2f", cmap="mako",
        square=True, mask=matrix.isna(),
        vmin=vmin_b, vmax=vmax_b,
        cbar=(ax == axes[1]),
        cbar_kws={"label": "Median binder RMSD (Å)"} if ax == axes[1] else {},
        linewidths=0.5, linecolor="white", annot_kws={"size": 8}
    )
    ax.set_title(title)
    ax.set_xlabel("Model 1"); ax.set_ylabel("Model 2")
    ax.tick_params(axis="x", rotation=45)
    ax.tick_params(axis="y", rotation=0)
    sns.despine(ax=ax, left=True, bottom=True)
fig.suptitle("Pair-wise Binder-aligned Cα RMSD \n Split by Experimental binding", y=1.03)
plt.tight_layout()
plt.savefig("Figure_8_binder_rmsd_heatmap_split.pdf", dpi=600, bbox_inches="tight")
plt.show()

# Per-method binder RMSD distribution
long_brmsd = pd.concat([
    plot_binder[["method_1", "rmsd_binder_aligned"]].rename(columns={"method_1": "method"}),
    plot_binder[["method_2", "rmsd_binder_aligned"]].rename(columns={"method_2": "method"})
]).reset_index(drop=True)

plt.figure(figsize=(10, 5))
sns.boxenplot(data=long_brmsd, x="method", y="rmsd_binder_aligned",
              order=[m for m in method_order if m in long_brmsd["method"].unique()],
              color="#C44E52", linewidth=1, k_depth="proportion")
plt.xlabel("Model")
plt.ylabel("Binder-aligned Cα RMSD (Å)")
plt.title("Distribution of pairwise binder aligned RMSD ")
plt.xticks(rotation=45, ha="right")
sns.despine()
plt.tight_layout()
plt.savefig("Figure_9_binder_rmsd_per_method.pdf", dpi=600, bbox_inches="tight")
plt.show()


In [ ]:

# ─────────────────────────────────────────────
# 7. COMBINED CONSENSUS + STATISTICS
# ─────────────────────────────────────────────
print("\nBuilding consensus dataframe...")

jaccard_summary = (
    jaccard_df
    .groupby("binder_sequence")
    .agg(
        median_jaccard=("jaccard", "median"),
        mean_jaccard=("jaccard", "mean"),
        binding=("binding", "first")
    )
    .reset_index()
)

pocket_summary = (
    plot_pocket
    .groupby("binder_sequence")
    .agg(
        median_pocket_rmsd=("rmsd_pocket_aligned", "median"),
        mean_pocket_rmsd=("rmsd_pocket_aligned", "mean")
    )
    .reset_index()
)

binder_summary = (
    plot_binder
    .groupby("binder_sequence")
    .agg(
        median_binder_rmsd=("rmsd_binder_aligned", "median"),
        mean_binder_rmsd=("rmsd_binder_aligned", "mean")
    )
    .reset_index()
)

stats_df = (
    jaccard_summary
    .merge(pocket_summary,  on="binder_sequence", how="left")
    .merge(binder_summary,  on="binder_sequence", how="left")
)
stats_df["binding"] = stats_df["binding"].astype(bool)

print(f"  Binders in consensus df: {len(stats_df)}")
print(f"  Binding True:  {stats_df['binding'].sum()}")
print(f"  Binding False: {(~stats_df['binding']).sum()}")

# ── 7a. Mann-Whitney U + effect size ─────────────────────────────────────────
def mannwhitney_with_effect_size(df, metric, group_col="binding"):
    binders     = df[df[group_col] == True][metric].dropna()
    non_binders = df[df[group_col] == False][metric].dropna()
    u_stat, p_value = mannwhitneyu(binders, non_binders, alternative="two-sided")
    n1, n2 = len(binders), len(non_binders)
    rank_biserial = (2 * u_stat) / (n1 * n2) - 1
    return {
        "metric": metric,
        "n_binders": n1,
        "n_non_binders": n2,
        "binder_median": binders.median(),
        "non_binder_median": non_binders.median(),
        "U": u_stat,
        "p_value": p_value,
        "rank_biserial": rank_biserial
    }

metrics = ["median_jaccard", "median_pocket_rmsd", "median_binder_rmsd"]
results = pd.DataFrame([mannwhitney_with_effect_size(stats_df, m) for m in metrics])
results["p_adj_fdr"] = multipletests(results["p_value"], method="fdr_bh")[1]
print("\nMann-Whitney results:")
print(results.to_string(index=False))

# ── 7b. Boxplot with significance annotations ─────────────────────────────────
def p_to_stars(p):
    if p < 0.001: return "***"
    elif p < 0.01: return "**"
    elif p < 0.05: return "*"
    else: return "ns"

metric_labels = {
    "median_jaccard":     "Median Jaccard index",
    "median_pocket_rmsd": "Median pocket-aligned RMSD",
    "median_binder_rmsd": "Median binder-aligned RMSD"
}

plot_df = stats_df.melt(
    id_vars=["binder_sequence", "binding"],
    value_vars=metrics,
    var_name="metric", value_name="value"
).reset_index(drop=True)
plot_df["metric"] = plot_df["metric"].map(metric_labels)

g = sns.catplot(
    data=plot_df,
    x="binding", y="value",
    hue="binding", col="metric",
    kind="box", sharey=False,
    palette=palette, hue_order=[False, True],
    order=[False, True], height=4, aspect=0.85,
    showfliers=False, legend=False
)
g.set_axis_labels("Experimental binding", "Value")
g.set_titles("{col_name}")

for ax in g.axes.flat:
    metric_name = ax.get_title()
    sns.stripplot(
        data=plot_df[plot_df["metric"] == metric_name],
        x="binding", y="value", order=[False, True],
        color="black", alpha=0.55, size=2.5, jitter=True, ax=ax
    )

stats_lookup = {metric_labels[row["metric"]]: row for _, row in results.iterrows()}

for ax in g.axes.flat:
    metric_name = ax.get_title()
    if metric_name not in stats_lookup:
        continue
    row = stats_lookup[metric_name]
    p, r_rb = row["p_value"], row["rank_biserial"]
    stars = p_to_stars(p)
    sub = plot_df[plot_df["metric"] == metric_name]
    y_max, y_min = sub["value"].max(), sub["value"].min()
    y_range = y_max - y_min
    y = y_max + 0.08 * y_range
    h = 0.04 * y_range
    ax.plot([0, 0, 1, 1], [y, y+h, y+h, y], color="black", linewidth=1)
    ax.text(0.5, y+h, stars, ha="center", va="bottom", fontsize=12, fontweight="bold")
    ax.text(0.5, y+h+0.08*y_range,
            f"p = {p:.2e}\nr\u1d63\u1d47 = {r_rb:.2f}",
            ha="center", va="bottom", fontsize=8)
    ax.set_ylim(y_min, y+h+0.25*y_range)

sns.despine()
plt.tight_layout()
plt.savefig("Figure_10_consensus_metrics_boxplot.pdf", dpi=600, bbox_inches="tight")
plt.show()



In [ ]:
# ─────────────────────────────────────────────
# 8. CONSENSUS SCORE
# ─────────────────────────────────────────────
print("\nBuilding consensus score...")

from sklearn.preprocessing import StandardScaler

model_df = stats_df.dropna(subset=["median_jaccard", "median_pocket_rmsd", "median_binder_rmsd", "binding"]).copy()

scaler = StandardScaler()
z = pd.DataFrame(
    scaler.fit_transform(model_df[["median_jaccard", "median_pocket_rmsd", "median_binder_rmsd"]]),
    columns=["median_jaccard_z", "median_pocket_rmsd_z", "median_binder_rmsd_z"],
    index=model_df.index
)
model_df = pd.concat([model_df, z], axis=1)

# Full 3-feature consensus score:
# +Jaccard  → higher interface overlap = more agreement = good
# -PocketRMSD → higher positional disagreement = bad
# -BinderRMSD → higher fold disagreement = bad
model_df["consensus_score"] = (
    model_df["median_jaccard_z"]
    - model_df["median_pocket_rmsd_z"]
    + model_df["median_binder_rmsd_z"]
)

# ── 8a. Consensus score boxplot ───────────────────────────────────────────────
plt.figure(figsize=(3.5, 4.5))
ax = sns.boxplot(
    data=model_df, x="binding", y="consensus_score",
    hue="binding", palette=palette,
    order=[False, True], hue_order=[False, True],
    showfliers=False, legend=False,
    width=0.55
)
sns.stripplot(
    data=model_df, x="binding", y="consensus_score",
    order=[False, True], color="black", alpha=0.55, size=2.5, jitter=True, ax=ax
)
ax.set_xlabel("Experimental binding")
ax.set_ylabel("Combined consensus score")
ax.set_title("Composite consensus performance \n based on experimental binding")
sns.despine()
plt.tight_layout()
plt.savefig("Figure_11_consensus_score_boxplot.pdf", dpi=600, bbox_inches="tight")
plt.show()

consensus_result = mannwhitney_with_effect_size(model_df, "consensus_score")
print("\nConsensus score Mann-Whitney:")
print(pd.Series(consensus_result))

# ── 8b. Non-parametric AUC bar chart ─────────────────────────────────────────
# AUROC derived directly from Mann-Whitney U: AUC = U / (n1 * n2)
# No model fitting — purely rank-based, equivalent to Wilcoxon AUROC.
from sklearn.metrics import roc_curve, auc, precision_recall_curve, average_precision_score

def mw_auc(df, metric, group_col="binding"):
    pos = df[df[group_col] == True][metric].dropna()
    neg = df[df[group_col] == False][metric].dropna()
    from scipy.stats import mannwhitneyu
    u, _ = mannwhitneyu(pos, neg, alternative="two-sided")
    raw_auc = u / (len(pos) * len(neg))
    # Flip if <0.5 (sign convention — metric may be inversely related)
    return max(raw_auc, 1 - raw_auc)

auc_metrics = {
    "Jaccard":       "median_jaccard",
    "Pocket RMSD":   "median_pocket_rmsd",
    "Binder RMSD":   "median_binder_rmsd",
    "Consensus\nscore": "consensus_score",
}

bar_data = []
for label, col in auc_metrics.items():
    a = mw_auc(model_df, col)
    # Bootstrap 95% CI
    rng = np.random.default_rng(42)
    boot = [mw_auc(model_df.sample(frac=1, replace=True, random_state=int(rng.integers(1e6))), col)
            for _ in range(1000)]
    bar_data.append({
        "metric": label,
        "auroc": a,
        "ci_lo": np.percentile(boot, 2.5),
        "ci_hi": np.percentile(boot, 97.5),
    })

bar_df = pd.DataFrame(bar_data)

bar_colors = ["#A8DADC", "#F4A261", "#B7E4C7", "#CDB4DB"]

fig, ax = plt.subplots(figsize=(5.5, 4))
bars = ax.bar(
    bar_df["metric"], bar_df["auroc"],
    color=bar_colors, edgecolor="black", linewidth=0.8, width=0.55, zorder=3
)
ax.errorbar(
    x=range(len(bar_df)),
    y=bar_df["auroc"],
    yerr=[bar_df["auroc"] - bar_df["ci_lo"], bar_df["ci_hi"] - bar_df["auroc"]],
    fmt="none", color="black", capsize=4, linewidth=1, zorder=4
)
ax.axhline(0.5, linestyle="--", color="#555555", linewidth=1, zorder=2)
ax.set_ylim(0, 1.0)
ax.set_ylabel("AUROC (non-parametric)", fontsize=12)
ax.set_xlabel("")
ax.set_title(
    "Predicting experimental binding from consensus metrics\n"
    "Non-parametric AUROC (Mann–Whitney, 95% CI bootstrap)",
    fontsize=11, fontweight="bold"
)
ax.tick_params(axis="x", length=0, labelsize=10)
ax.tick_params(axis="y", labelsize=10)
ax.text(3.55, 0.505, "AUROC=0.50",color="#555555", fontsize=8, va="bottom")
ax.yaxis.grid(True, linestyle=":", linewidth=0.6, color="lightgray", zorder=0)
ax.set_axisbelow(True)
sns.despine(ax=ax)
plt.tight_layout()
plt.savefig("Figure_12_auroc_barplot.pdf", dpi=600, bbox_inches="tight")
plt.show()

# ── 8c. ROC curve (consensus score only, no model) ────────────────────────────
y      = model_df["binding"].astype(int)
y_score = model_df["consensus_score"].values

# ROC
fpr, tpr, _ = roc_curve(y, y_score)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(5, 5))
plt.plot(fpr, tpr, linewidth=2, label=f"ROC AUC = {roc_auc:.2f}")
plt.plot([0, 1], [0, 1], linestyle="--", linewidth=1, color="gray")
plt.xlabel("False positive rate")
plt.ylabel("True positive rate / Recall")
plt.title("ROC curve: binding prediction")
plt.legend(frameon=False, loc="lower right")
sns.despine()
plt.tight_layout()
plt.savefig("Figure_12_roc_curve.pdf", dpi=600, bbox_inches="tight")
plt.show()

# PR
precision, recall, _ = precision_recall_curve(y, y_score)
avg_precision = average_precision_score(y, y_score)
baseline = y.mean()

plt.figure(figsize=(5, 5))
plt.plot(recall, precision, linewidth=2, label=f"Average precision = {avg_precision:.2f}")
plt.axhline(baseline, linestyle="--", linewidth=1, color="gray", label=f"Baseline = {baseline:.2f}")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision–recall curve: binding prediction")
plt.legend(frameon=False, loc="upper right")
sns.despine()
plt.tight_layout()
plt.savefig("Figure_13_pr_curve.pdf", dpi=600, bbox_inches="tight")
plt.show()


In [ ]:

# ─────────────────────────────────────────────
# 9. NORMALIZED TARGET CONTACT HEATMAP
# ─────────────────────────────────────────────
print("\nGenerating target contact distribution heatmap...")

method_residue_matrix = (
    plot_target
    .groupby(["method", "target_resseq"])
    .size()
    .unstack(fill_value=0)
)

contact_fraction_matrix = (
    method_residue_matrix
    .div(method_residue_matrix.sum(axis=1), axis=0)
    .fillna(0)
)

threshold = 0.002
plot_matrix = contact_fraction_matrix.where(contact_fraction_matrix >= threshold)
plot_matrix = plot_matrix.loc[:, plot_matrix.notna().any(axis=0)]

plt.figure(figsize=(14, 5))
sns.heatmap(
    plot_matrix, cmap="mako",
    mask=plot_matrix.isna(),
    cbar_kws={"label": "Fraction of contacts"},
    linewidths=0.5, linecolor="white"
)
plt.xlabel("Target residue position")
plt.ylabel("Method")
plt.title("Normalized target contact distribution across methods")
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
sns.despine(left=True, bottom=True)
plt.tight_layout()
plt.savefig("Figure_14_target_contact_heatmap.pdf", dpi=600, bbox_inches="tight")
plt.show()

print("\nAll done! Plots saved as PDF files in the working directory.")



In [ ]:
# ─────────────────────────────────────────────
# 10. CHERRY-PICKED BINDERS — PUBLICATION FIGURE
# ─────────────────────────────────────────────
# Self-contained: data hardcoded from cherry-picked summary table.
# Run as standalone cell — no dependency on earlier variables.
# ─────────────────────────────────────────────
print("\nGenerating cherry-picked binders publication figure...")
 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import matplotlib.ticker as ticker
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns
 
sns.set_theme(style="ticks", context="paper", font_scale=1.15)
plt.rcParams.update({"pdf.fonttype": 42, "ps.fonttype": 42,
                     "figure.dpi": 300, "savefig.dpi": 600})
 
# ── Hardcoded cherry data (from summary table) ────────────────────────────────
cherry_data = {
    "binder_name":      ["dark-moth-cypress", "small-fox-pearl",
                         "golden-dove-snow",  "vast-bear-marble",
                         "green-eagle-lotus"],
    "designMethod":     ["BindCraft",  "BindCraft",    "RFdiffusion", "Unknown",    "Antiberty"],
    "binding":          [False,         True,           False,          True,          True],
    "category":         ["False positive\n(AF-biased)",
                         "True positive\n(high consensus)",
                         "False positive\n(low OF/Chai)",
                         "True positive\n(AF3 only)",
                         "True positive\n(all models low)"],
    "af2_iptm":         [0.9127,  0.9239,  0.9222,  np.nan,  0.5159],
    "af3_iptm":         [0.9200,  0.9100,  0.9100,  0.8800,  0.1500],
    "of3_iptm":         [0.9263,  0.9032,  0.1890,  0.1519,  0.2889],
    "chai1_iptm":       [0.7863,  0.7424,  0.2371,  0.2106,  0.2332],
    "helixfold3_iptm":  [0.7859,  0.8954,  0.1534,  0.8103,  0.2057],
    "median_jaccard":   [0.7589,  0.8438,  0.1491,  0.1894,  0.1936],
    "consensus_score":  [1.9614,  2.3295, -3.0028, -3.3071, -3.0911],
}
 
cdf = pd.DataFrame(cherry_data).set_index("binder_name")
cherry = list(cdf.index)
 
score_cols  = ["af2_iptm", "af3_iptm", "of3_iptm",
               "chai1_iptm", "helixfold3_iptm", "median_jaccard"]
col_labels  = ["AF2\nipTM", "AF3\nipTM", "OF3\nipTM",
               "Chai-1\nipTM", "HelixFold3\nipTM", "Median\nJaccard"]
 
matrix = cdf[score_cols].astype(float).values   # (5, 6)
 
# ── Colours ───────────────────────────────────────────────────────────────────
BINDER_COL    = "#2e7d32"   # forest green
NONBINDER_COL = "#c62828"   # crimson
nature_cmap   = LinearSegmentedColormap.from_list(
    "nature_rg", ["#c62828", "#f5f5f0", "#2e7d32"], N=256
)
 
# ── Layout: heatmap (left) + consensus bar (right) ───────────────────────────
fig = plt.figure(figsize=(14, 5.5))
gs  = gridspec.GridSpec(1, 2, width_ratios=[3.2, 1],
                        left=0.01, right=0.97,
                        bottom=0.08, top=0.88, wspace=0.06)
 
ax_heat = fig.add_subplot(gs[0])
ax_bar  = fig.add_subplot(gs[1])
 
# ════════════════════════════════════════════════════════
# LEFT PANEL — score heatmap
# ════════════════════════════════════════════════════════
n_rows, n_cols = matrix.shape
 
im = ax_heat.imshow(matrix, cmap=nature_cmap, vmin=0, vmax=1,
                    aspect="auto", interpolation="nearest")
 
# white cell borders
ax_heat.set_xticks([]); ax_heat.set_yticks([])
for i in range(n_rows + 1):
    ax_heat.axhline(i - 0.5, color="white", linewidth=2.5, zorder=3)
for j in range(n_cols + 1):
    ax_heat.axvline(j - 0.5, color="white", linewidth=2.5, zorder=3)
 
# cell value annotations
for i in range(n_rows):
    for j in range(n_cols):
        val = matrix[i, j]
        if np.isnan(val):
            txt, fc = "N/A", "#999999"
        else:
            txt = f"{val:.2f}"
            fc  = "white" if (val < 0.25 or val > 0.72) else "#1a1a1a"
        ax_heat.text(j, i, txt, ha="center", va="center",
                     fontsize=9.5, fontweight="semibold", color=fc, zorder=4)
 
# column labels (top)
ax_heat.set_xticks(range(n_cols))
ax_heat.set_xticklabels(col_labels, fontsize=10.5, fontweight="bold",
                         ha="center", va="bottom")
ax_heat.xaxis.set_ticks_position("top")
ax_heat.xaxis.set_label_position("top")
ax_heat.tick_params(axis="x", length=0, pad=5)
 
# row labels
row_labels = []
for b in cherry:
    row     = cdf.loc[b]
    outcome = " Binder" if row["binding"] else " Non-binder"
    row_labels.append(f"{b}  | {outcome}  |  {row['designMethod']}")
 
ax_heat.set_yticks(range(n_rows))
ax_heat.set_yticklabels(row_labels, fontsize=9, fontfamily="monospace")
ax_heat.tick_params(axis="y", length=0, pad=32)
 
# binding outcome strip (left margin)
for i, b in enumerate(cherry):
    fc = BINDER_COL if cdf.loc[b, "binding"] else NONBINDER_COL
    ax_heat.add_patch(plt.Rectangle(
        (-0.60, i - 0.44), 0.08, 0.88,
        color=fc, transform=ax_heat.transData,
        clip_on=False, zorder=5, linewidth=0
    ))
 
# colorbar
cbar = fig.colorbar(im, ax=ax_heat, fraction=0.022, pad=0.012, aspect=25)
cbar.set_label("Normalised score", fontsize=9.5, labelpad=6)
cbar.set_ticks([0, 0.25, 0.5, 0.75, 1.0])
cbar.set_ticklabels(["0.00\n(low)", "0.25", "0.50", "0.75", "1.00\n(high)"])
cbar.ax.tick_params(labelsize=8.5, length=0)
cbar.outline.set_linewidth(0)
 
for spine in ax_heat.spines.values():
    spine.set_visible(False)
 
# ════════════════════════════════════════════════════════
# RIGHT PANEL — consensus score bar chart
# ════════════════════════════════════════════════════════
scores   = cdf["consensus_score"].values
bindings = cdf["binding"].values
bar_cols = [BINDER_COL if b else NONBINDER_COL for b in bindings]
 
# horizontal bars, sorted by score
order    = np.argsort(scores)
y_pos    = np.arange(n_rows)
 
bars = ax_bar.barh(
    y_pos, scores[order],
    color=[bar_cols[i] for i in order],
    edgecolor="white", linewidth=0.8,
    height=0.6, zorder=3
)
 
# zero line
ax_bar.axvline(0, color="#444444", linewidth=1.0, zorder=4)
 
# value labels inside/outside bars
for idx, (yp, sc) in enumerate(zip(y_pos, scores[order])):
    ha  = "left"  if sc >= 0 else "right"
    xoff = 0.08 if sc >= 0 else -0.08
    ax_bar.text(sc + xoff, yp, f"{sc:+.2f}",
                va="center", ha=ha, fontsize=8.5,
                fontweight="semibold",
                color="#1a1a1a")
 
# y-tick labels = short binder name only
ax_bar.set_yticks(y_pos)
ax_bar.set_yticklabels(
    [cherry[i] for i in order],
    fontsize=8.5, fontfamily="monospace"
)
ax_bar.tick_params(axis="y", length=0)
ax_bar.tick_params(axis="x", labelsize=8.5)
ax_bar.set_xlabel("Consensus score\n(Jaccard_z − PocketRMSD_z − BinderRMSD_z)",
                   fontsize=9)
ax_bar.yaxis.set_ticks_position("right")
ax_bar.yaxis.set_label_position("right")
ax_bar.set_xlim(scores.min() - 1.2, scores.max() + 1.5)
ax_bar.xaxis.grid(True, linestyle=":", linewidth=0.6,
                   color="lightgray", zorder=0)
ax_bar.set_axisbelow(True)
sns.despine(ax=ax_bar, left=True)
 
# ── Shared legend ─────────────────────────────────────────────────────────────
patch_b  = mpatches.Patch(facecolor=BINDER_COL,    edgecolor="none",
                            label="Experimental binder  ✓")
patch_nb = mpatches.Patch(facecolor=NONBINDER_COL, edgecolor="none",
                            label="Non-binder  ✗")
fig.legend(
    handles=[patch_b, patch_nb],
    loc="lower center", bbox_to_anchor=(0.5, -0.04),
    ncol=2, frameon=True, framealpha=0.95,
    edgecolor="#cccccc", fontsize=9.5,
    handlelength=1.2, handleheight=1.2, columnspacing=1.5
)
 
# ── Figure title + subtitle ───────────────────────────────────────────────────
fig.suptitle(
    "Cherry-picked binders: per-model confidence scores vs experimental binding outcome",
    fontsize=12.5, fontweight="bold", y=0.98, x=0.40
)
fig.text(
    0.40, 0.91,
    "Left: per-model iPTM and Jaccard scores (green = high confidence, red = low).  "
    "Right: composite consensus score.  "
    "Side strip = experimental binding outcome.",
    fontsize=8.5, color="#555555", ha="center"
)
 
plt.savefig("Figure_15_cherry_heatmap.pdf", dpi=600, bbox_inches="tight")
plt.savefig("Figure_15_cherry_heatmap.png", dpi=300, bbox_inches="tight")
plt.show()
print("  Cherry figure saved.")
 
 


In [ ]:
# ─────────────────────────────────────────────
# CHERRY-PICKED BINDERS — HEATMAP ONLY
# Publication-ready version
# ─────────────────────────────────────────────

print("\nGenerating cherry-picked binders heatmap...")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns

sns.set_theme(style="ticks", context="paper", font_scale=1.2)

plt.rcParams.update({
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "figure.dpi": 300,
    "savefig.dpi": 600
})

# ─────────────────────────────────────────────
# Data
# ─────────────────────────────────────────────

cherry_data = {
    "binder_name": [
        "dark-moth-cypress",
        "small-fox-pearl",
        "golden-dove-snow",
        "green-eagle-lotus"
    ],

    "binding": [
        False,
        True,
        False,
        True
    ],

    "af2_iptm": [
        0.9127,
        0.9239,
        0.9222,
        0.5159
    ],

    "af3_iptm": [
        0.9200,
        0.9100,
        0.9100,
        0.1500
    ],

    "of3_iptm": [
        0.9263,
        0.9032,
        0.1890,
        0.2889
    ],

    "chai1_iptm": [
        0.7863,
        0.7424,
        0.2371,
        0.2332
    ],

    "helixfold3_iptm": [
        0.7859,
        0.8954,
        0.1534,
        0.2057
    ],

    "median_jaccard": [
        0.7589,
        0.8438,
        0.1491,
        0.1936
    ]
}

cdf = pd.DataFrame(cherry_data).set_index("binder_name")

score_cols = [
    "af2_iptm",
    "af3_iptm",
    "of3_iptm",
    "chai1_iptm",
    "helixfold3_iptm",
    "median_jaccard"]

col_labels = [
    "AF2\nipTM",
    "AF3\nipTM",
    "OF3\nipTM",
    "Chai-1\nipTM",
    "HelixFold3\nipTM",
    
    "Median\nJaccard"
]

matrix = cdf[score_cols].values

# ─────────────────────────────────────────────
# Colors
# ─────────────────────────────────────────────

BINDER_COL = "#2e7d32"
NONBINDER_COL = "#c62828"

nature_cmap = LinearSegmentedColormap.from_list(
    "nature_rg",
    ["#c62828", "#f5f5f0", "#2e7d32"],
    N=256
)

# ─────────────────────────────────────────────
# Figure
# ─────────────────────────────────────────────

fig, ax_heat = plt.subplots(
    figsize=(14.5, 6.2)
)

fig.subplots_adjust(
    left=0.24,
    right=0.90,
    bottom=0.14,
    top=0.84
)

# ─────────────────────────────────────────────
# Heatmap
# ─────────────────────────────────────────────

im = ax_heat.imshow(
    matrix,
    cmap=nature_cmap,
    vmin=0,
    vmax=1,
    aspect="auto",
    interpolation="nearest"
)

n_rows, n_cols = matrix.shape

# white borders

for i in range(n_rows + 1):
    ax_heat.axhline(
        i - 0.5,
        color="white",
        linewidth=2.8,
        zorder=3
    )

for j in range(n_cols + 1):
    ax_heat.axvline(
        j - 0.5,
        color="white",
        linewidth=2.8,
        zorder=3
    )

# ─────────────────────────────────────────────
# Cell values
# ─────────────────────────────────────────────

for i in range(n_rows):
    for j in range(n_cols):

        val = matrix[i, j]

        txt = f"{val:.2f}"

        color = (
            "white"
            if (val < 0.25 or val > 0.72)
            else "#1a1a1a"
        )

        ax_heat.text(
            j,
            i,
            txt,
            ha="center",
            va="center",
            fontsize=11,
            fontweight="bold",
            color=color
        )

# ─────────────────────────────────────────────
# Column labels
# ─────────────────────────────────────────────

ax_heat.set_xticks(range(n_cols))

ax_heat.set_xticklabels(
    col_labels,
    fontsize=12,
    fontweight="bold"
)

ax_heat.xaxis.set_ticks_position("top")
ax_heat.xaxis.set_label_position("top")

ax_heat.tick_params(
    axis="x",
    length=0,
    pad=10
)

# ─────────────────────────────────────────────
# Row labels
# ─────────────────────────────────────────────
design_map = {
    "dark-moth-cypress": "BindCraft",
    "small-fox-pearl": "BindCraft",
    "golden-dove-snow": "RFdiffusion",
    "green-eagle-lotus": "Antiberty"
}


row_labels = []

for binder in cdf.index:
    design_method = design_map.get(binder, "Unknown")
    row_labels.append(f"{binder}  |  {design_method}")

ax_heat.set_yticks(range(n_rows))

ax_heat.set_yticklabels(
    row_labels,
    fontsize=11,
    fontfamily="Arial"
)

ax_heat.tick_params(
    axis="y",
    length=0,
    pad=25
)

# ─────────────────────────────────────────────
# Experimental outcome strip
# ─────────────────────────────────────────────

for i, binder in enumerate(cdf.index):

    color = (
        BINDER_COL
        if cdf.loc[binder, "binding"]
        else NONBINDER_COL
    )

    ax_heat.add_patch(
        plt.Rectangle(
            (-0.62, i - 0.44),
            0.08,
            0.88,
            color=color,
            transform=ax_heat.transData,
            clip_on=False,
            linewidth=0
        )
    )

# ─────────────────────────────────────────────
# Colorbar
# ─────────────────────────────────────────────

cbar = fig.colorbar(
    im,
    ax=ax_heat,
    fraction=0.03,
    pad=0.015
)

cbar.set_label(
    "Normalised score",
    fontsize=11
)

cbar.set_ticks(
    [0, 0.25, 0.50, 0.75, 1.00]
)

cbar.set_ticklabels(
    [
        "0\n(low)",
        "0.25",
        "0.50",
        "0.75",
        "1\n(high)"
    ]
)

cbar.ax.tick_params(
    labelsize=10,
    length=0
)

cbar.outline.set_visible(False)

# ─────────────────────────────────────────────
# Clean axes
# ─────────────────────────────────────────────

for spine in ax_heat.spines.values():
    spine.set_visible(False)

# ─────────────────────────────────────────────
# Legend
# ─────────────────────────────────────────────

patch_b = mpatches.Patch(
    facecolor=BINDER_COL,
    edgecolor="none",
    label="Experimental binder"
)

patch_nb = mpatches.Patch(
    facecolor=NONBINDER_COL,
    edgecolor="none",
    label=" Experimental non-binder"
)

fig.legend(
    handles=[patch_b, patch_nb],
    loc="lower center",
    bbox_to_anchor=(0.5, -0.02),
    ncol=2,
    frameon=False,
    fontsize=11
)

# ─────────────────────────────────────────────
# Title
# ─────────────────────────────────────────────

fig.suptitle(
    "Cherry-picked Examples: Per-model confidence scores vs experimental binding outcome",
    fontsize=16,
    fontweight="bold",
    y=0.97
)



# ─────────────────────────────────────────────
# Save
# ─────────────────────────────────────────────

plt.show()

print("✓ Figure saved.")

In [ ]:
import pandas as pd

df2 = pd.read_parquet(r"C:\\Users\\nehaj\\OneDrive\\Desktop\\Block_4_new_project\\a_contest_2\\prediction_records_with_metrics.parquet")
df1 = pd.read_parquet(r"C:\\Users\\nehaj\\OneDrive\\Desktop\\Block_4_new_project\\prediction_records_with_metrics.parquet")

df1["contest"] = 1
df2["contest"] = 2

df_merged = pd.concat([df1, df2], ignore_index=True)
df_merged.to_parquet("merged_records.parquet", index=False)
# Check if the dataframe contains ANY NaN values
